In [2]:
import sqlite3
import pandas as pd

DB_PATH = "../DB/oedb_baseline.db"
service_account_file = "../config/service_account_key.json"

from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor

file_loader = GoogleDriveLoader(service_account_file)
extractor = TextExtractor()

rows = []

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    for notegroup_id in range(64, 83):
        cursor.execute("""
            SELECT project_name, phase, note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()

        if row is None:
            print(f"[{notegroup_id}] No notegroup found — skipping")
            continue

        project_name, phase, note_url_qa, note_url_participant = row
        all_drive_paths = {}

        for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
            if not url:
                continue
            try:
                result = file_loader.load(url)
                all_drive_paths[label] = result['drive_path']
            except Exception as e:
                print(f"[{notegroup_id}] {label} load failed: {e}")

        combined_drive_paths = "|".join(all_drive_paths.values())
        rows.append({"notegroupID": notegroup_id, "combined_drive_paths": combined_drive_paths})

df = pd.DataFrame(rows)

Loading: Liedscham notite_IYAD.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/Liedscham notite_IYAD.docx
Loading: Deelnemers geselecteerd en uitgenodigd (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/1370 Werving en Deelnemers/Deelnemers geselecteerd en uitgenodigd
Loading: MAHAD_NOTITIES.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/MAHAD_NOTITIES.docx
Loading: Deelnemers geselecteerd en ui

In [5]:
pd.set_option("display.max_colwidth", None)
df

,notegroupID,combined_drive_paths
0,64,"01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/Liedscham notite_IYAD.docx|01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/1370 Werving en Deelnemers/Deelnemers geselecteerd en uitgenodigd"
1,65,"01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/MAHAD_NOTITIES.docx|01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/1370 Werving en Deelnemers/Deelnemers geselecteerd en uitgenodigd"
2,66,"01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/NOTES FATIH Groep_gesprek_vragen_Leidschendam.docx|01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/1370 Werving en Deelnemers/Deelnemers geselecteerd en uitgenodigd"
3,67,"01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/Notes FLORIS 1370|01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/1370 Werving en Deelnemers/Deelnemers geselecteerd en uitgenodigd"
4,68,"01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/Notes/Reza_notities.docx|01 extern/01 huidige klanten : projecten/ ARCHIEF /1370 - Evaluatie Inburgeringsbeleid Leidschendam-Voorburg, Wassenaar en Voorschoten/Interviews_met_de deelnemers/1370 Werving en Deelnemers/Deelnemers geselecteerd en uitgenodigd"
5,69,01 extern/01 huidige klanten : projecten/1377 - Expertpool Inburgering Utrecht (1 sessie)/expertpool/Data/1377 Note-taking template (Ingrid)
6,70,01 extern/01 huidige klanten : projecten/1377 - Expertpool Inburgering Utrecht (1 sessie)/expertpool/Data/1377 Note-taking template Iyad
7,71,01 extern/01 huidige klanten : projecten/1377 - Expertpool Inburgering Utrecht (1 sessie)/expertpool/Data/Ingrid - Interviews 4 February
8,72,01 extern/01 huidige klanten : projecten/1377 - Expertpool Inburgering Utrecht (1 sessie)/expertpool/Data/Notes Sandrine
9,73,01 extern/01 huidige klanten : projecten/1377 - Expertpool Inburgering Utrecht (1 sessie)/expertpool/Data/Reza 1377 Note-taking template - interview.docx
